# Segmentation d'images avec U-Net et PyTorch

La **segmentation sémantique** est une tâche fondamentale en vision par ordinateur : au lieu d'attribuer une seule étiquette à toute une image (classification), on attribue une classe à **chaque pixel** de l'image.

**Applications :**
- **Médecine** : segmentation de tumeurs, d'organes sur des IRM ou scanners
- **Conduite autonome** : délimitation de routes, piétons, véhicules
- **Satellite** : analyse de terrain, détection de bâtiments
- **Robotique** : compréhension de la scène environnante

Dans ce notebook, vous allez implémenter **U-Net**, l'architecture de référence pour la segmentation, et l'entraîner pour localiser précisément les animaux de compagnie dans des images.

## L'architecture U-Net

L'architecture U-Net a été introduite en 2015 par Ronneberger et al. pour la segmentation d'images médicales. Elle tire son nom de sa forme en **"U"** et repose sur deux chemins symétriques.

### Chemin contractant — Encodeur
Similaire à un CNN classique, il réduit progressivement la résolution spatiale tout en augmentant le nombre de canaux. Chaque étape comprend :
- Deux convolutions 3×3 + BatchNorm + ReLU (**DoubleConv**)
- Un Max Pooling 2×2 pour diviser la résolution par 2

### Chemin expansif — Décodeur
Le décodeur reconstruit progressivement la résolution originale. Chaque étape comprend :
- Une **convolution transposée** (stride=2) pour doubler la résolution
- La **concaténation** avec les feature maps correspondantes de l'encodeur (skip connections)
- Deux convolutions 3×3 + BatchNorm + ReLU

### Skip Connections (connexions de saut)
C'est l'innovation clé de U-Net : les feature maps de l'encodeur sont **directement connectées** aux feature maps correspondantes du décodeur. Cela permet de combiner les informations de **haute résolution** (localisation précise) avec les informations **sémantiques** (contexte global).

```
Entrée [3 × 128 × 128]
│
DoubleConv → [16 × 128 × 128] ─────────────────────── (skip 1) ───┐
│                                                                   │
MaxPool → 64 × 64                                          concat + DoubleConv → Sortie
│                                                                   │
DoubleConv → [32 × 64 × 64] ────────────── (skip 2) ───┐   ConvTranspose
│                                                        │
MaxPool → 32 × 32                               concat + DoubleConv
│                                                        │
DoubleConv → [64 × 32 × 32] ──── (skip 3) ───┐   ConvTranspose
│                                              │
MaxPool → 16 × 16                    concat + DoubleConv
│                                              │
DoubleConv → [128 × 16 × 16] ─────────   ConvTranspose
       (Bottleneck)
```

## Importation des bibliothèques

In [ ]:
import torch
from torch import nn
import torchvision
from torchvision import transforms, datasets
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader, random_split

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device utilisé : {device}")

## Importation des données

Nous allons utiliser le dataset **Oxford-IIIT Pet**, qui contient ~7 400 images de chats et chiens annotées avec des masques de segmentation pixel par pixel.

Chaque masque contient trois valeurs :
- **1** : pixels appartenant à l'animal (avant-plan)
- **2** : pixels d'arrière-plan
- **3** : pixels de contour/ambigus

Nous simplifions en segmentation **binaire** : animal (1) vs. reste (0).

In [ ]:
class SegmentationDataset(torch.utils.data.Dataset):
    def __init__(self, root, split='trainval', img_size=128):
        self.base_dataset = torchvision.datasets.OxfordIIITPet(
            root=root,
            split=split,
            target_types='segmentation',
            download=True
        )
        self.img_size = img_size
        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, mask = self.base_dataset[idx]
        img = self.img_transform(img)

        mask = np.array(mask.resize((self.img_size, self.img_size), Image.NEAREST))
        # 1 = animal → avant-plan, 2 et 3 → arrière-plan
        mask = (mask == 1).astype(np.float32)
        mask = torch.tensor(mask).unsqueeze(0)

        return img, mask

Créez le dataset avec `SegmentationDataset(root='./data')`.

Divisez-le en ensemble d'entraînement (80%) et de validation (20%) avec `random_split`.

Créez deux DataLoaders (batch_size=16) :
- `dataloader_train` : shuffle activé
- `dataloader_val` : shuffle désactivé

In [ ]:
dataset = SegmentationDataset(root='./data')

# À compléter : diviser en train (80%) et val (20%) avec random_split
dataset_train, dataset_val = None, None

# À compléter : créer les DataLoaders (batch_size=16)
dataloader_train = None
dataloader_val   = None

In [ ]:
# Visualisation de quelques exemples
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

for i in range(3):
    img, mask = dataset[i * 500]
    img_display = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

    axes[i, 0].imshow(img_display)
    axes[i, 0].set_title("Image originale")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(mask.squeeze(), cmap='gray')
    axes[i, 1].set_title("Masque de segmentation")
    axes[i, 1].axis('off')

    overlay = img_display.copy()
    mask_np = mask.squeeze().numpy()
    overlay[mask_np == 1] = overlay[mask_np == 1] * 0.4 + np.array([0.0, 0.8, 0.0]) * 0.6
    axes[i, 2].imshow(overlay.clip(0, 1))
    axes[i, 2].set_title("Superposition")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## Les blocs de base du U-Net

Avant de construire le U-Net complet, vous allez implémenter ses briques élémentaires une par une.

### Exercice 1 : Le bloc DoubleConv

Le `DoubleConv` est la brique fondamentale du U-Net. Il applique **deux fois** la séquence :
> Conv2d (3×3, padding=1) → BatchNorm2d → ReLU

Le `padding=1` garantit que les dimensions spatiales (H, W) sont **conservées** après chaque convolution.

Implémentez le bloc `DoubleConv` :
- Dans `__init__` : créez `self.double_conv` avec `nn.Sequential`
- Dans `forward` : appliquez `self.double_conv` sur l'entrée `x`

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = None  # À compléter avec nn.Sequential(Conv, BN, ReLU, Conv, BN, ReLU)

    def forward(self, x):
        return None  # À compléter

In [ ]:
# Vérification : le DoubleConv doit conserver H et W
test_input = torch.randn(2, 3, 64, 64)
test_block = DoubleConv(3, 16)
test_output = test_block(test_input)
print(f"Entrée : {test_input.shape}")   # [2, 3, 64, 64]
print(f"Sortie : {test_output.shape}")  # [2, 16, 64, 64]
assert test_output.shape == (2, 16, 64, 64), "Erreur : dimensions incorrectes !"
print("DoubleConv OK !")

### Exercice 2 : Le bloc Down (encodeur)

Le bloc `Down` représente **une étape de l'encodeur**. Il divise la résolution spatiale par 2 via un Max Pooling, puis applique un `DoubleConv`.

```
x → MaxPool2d(2) → DoubleConv(in_channels, out_channels) → sortie
```

Implémentez `Down` en utilisant `nn.Sequential` avec `nn.MaxPool2d(2)` et `DoubleConv`.

In [ ]:
class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = None  # À compléter : nn.Sequential(MaxPool, DoubleConv)

    def forward(self, x):
        return None  # À compléter

In [ ]:
# Vérification : Down doit diviser H et W par 2
test_input = torch.randn(2, 16, 64, 64)
test_block = Down(16, 32)
test_output = test_block(test_input)
print(f"Entrée : {test_input.shape}")   # [2, 16, 64, 64]
print(f"Sortie : {test_output.shape}")  # [2, 32, 32, 32]
assert test_output.shape == (2, 32, 32, 32), "Erreur : dimensions incorrectes !"
print("Down OK !")

### Exercice 3 : Le bloc Up (décodeur)

Le bloc `Up` représente **une étape du décodeur**. Il prend **deux entrées** :
- `x1` : feature map venant du bas (basse résolution, ex: 16×16, 128 canaux)
- `x2` : skip connection de l'encodeur correspondant (haute résolution, ex: 32×32, 64 canaux)

Fonctionnement pour `Up(128, 64)` :
```
x1 [128 × 16 × 16]
  → ConvTranspose2d(128, 64, k=2, stride=2)
  → x1_up [64 × 32 × 32]
  → cat([x2, x1_up], dim=1)      # x2 : [64 × 32 × 32]
  → [128 × 32 × 32]
  → DoubleConv(128, 64)
  → sortie [64 × 32 × 32]
```

Implémentez `Up` :
- `self.up` : `ConvTranspose2d(in_channels, in_channels//2, kernel_size=2, stride=2)`
- `self.conv` : `DoubleConv(in_channels, out_channels)`
- Dans `forward` : upsample x1, concaténer avec x2, appliquer self.conv

In [ ]:
class Up(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up   = None  # À compléter : ConvTranspose2d
        self.conv = None  # À compléter : DoubleConv

    def forward(self, x1, x2):
        # x1 : feature map du décodeur (basse résolution)
        # x2 : skip connection de l'encodeur (haute résolution)
        return None  # À compléter

In [ ]:
# Vérification : Up doit doubler H et W via la skip connection
x1 = torch.randn(2, 128, 16, 16)  # basse résolution (décodeur)
x2 = torch.randn(2, 64,  32, 32)  # haute résolution (skip connection)
test_block = Up(128, 64)
test_output = test_block(x1, x2)
print(f"x1 (décodeur)   : {x1.shape}")          # [2, 128, 16, 16]
print(f"x2 (skip conn.) : {x2.shape}")          # [2, 64, 32, 32]
print(f"Sortie          : {test_output.shape}") # [2, 64, 32, 32]
assert test_output.shape == (2, 64, 32, 32), "Erreur : dimensions incorrectes !"
print("Up OK !")

## Exercice 4 : L'architecture U-Net complète

Assemblez maintenant le U-Net complet à partir des blocs `DoubleConv`, `Down`, `Up`.

| Attribut | Bloc | Canaux (in → out) | Résolution |
|----------|------|-------------------|-----------|
| `inc` | DoubleConv | 3 → 16 | 128×128 |
| `down1` | Down | 16 → 32 | 64×64 |
| `down2` | Down | 32 → 64 | 32×32 |
| `down3` | Down | 64 → 128 | 16×16 |
| `up1` | Up | 128 → 64 | 32×32 |
| `up2` | Up | 64 → 32 | 64×64 |
| `up3` | Up | 32 → 16 | 128×128 |
| `outc` | Conv2d(k=1) | 16 → n_classes | 128×128 |

**Indice pour `forward` :** Sauvegardez x1, x2, x3, x4 (sorties de l'encodeur) et passez-les comme skip connections aux blocs Up.

N'héstiez pas à checker la [doc](https://docs.pytorch.org/docs/2.12/generated/torch.concat.html)

In [ ]:
class UNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=1):
        super().__init__()
        self.inc   = None  # DoubleConv(3 → 16)
        self.down1 = None  # Down(16 → 32)
        self.down2 = None  # Down(32 → 64)
        self.down3 = None  # Down(64 → 128)
        self.up1   = None  # Up(128 → 64)
        self.up2   = None  # Up(64 → 32)
        self.up3   = None  # Up(32 → 16)
        self.outc  = None  # Conv2d(16 → n_classes, kernel=1)

    def forward(self, x):
        # Encodeur : sauvegarder chaque sortie pour les skip connections
        # Décodeur : passer les skip connections aux blocs Up
        return None  # À compléter

In [ ]:
# Vérification : la sortie doit avoir la même résolution que l'entrée
model_test = UNet(n_channels=3, n_classes=1)
test_input = torch.randn(2, 3, 128, 128)
test_output = model_test(test_input)
print(f"Entrée : {test_input.shape}")   # [2, 3, 128, 128]
print(f"Sortie : {test_output.shape}")  # [2, 1, 128, 128]
assert test_output.shape == (2, 1, 128, 128), "Erreur : dimensions incorrectes !"
print("U-Net OK !")

n_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f"Nombre de paramètres entraînables : {n_params:,}")

## Fonctions de perte et métriques

En segmentation, les métriques classiques de classification (accuracy) sont peu adaptées car les classes peuvent être très déséquilibrées. On utilise des métriques plus robustes aux déséquilibres.

### Exercice 5 : La Dice Loss

Le **coefficient de Dice** mesure la similarité entre deux masques :

$$\text{Dice} = \frac{2 \times |A \cap B|}{|A| + |B|}$$

- Dice = 1 → prédiction parfaite
- Dice = 0 → aucune intersection

La **Dice Loss** = `1 - Dice` (à minimiser pendant l'entraînement).

Implémentez `DiceLoss.forward` :
1. Appliquer `torch.sigmoid` sur `prediction` (convertit les logits en probabilités)
2. Aplatir `prediction` et `target` avec `.view(-1)`
3. Calculer `intersection = (prediction * target).sum()`
4. Calculer `dice = (2 * intersection + smooth) / (prediction.sum() + target.sum() + smooth)`
5. Retourner `1 - dice`

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, prediction, target):
        # 1. Appliquer sigmoid sur prediction
        # 2. Aplatir prediction et target en 1D
        # 3. Calculer l'intersection et le coefficient de Dice
        # 4. Retourner 1 - dice
        return None  # À compléter

### Exercice 6 : La métrique IoU (Intersection over Union)

L'**IoU** est la métrique standard en segmentation :

$$\text{IoU} = \frac{|A \cap B|}{|A \cup B|}$$

Pour calculer l'IoU d'une prédiction continue, il faut d'abord la **seuiller** : si `sigmoid(logit) > 0.5`, le pixel est prédit comme avant-plan.

Implémentez `iou_score` :
1. Appliquer `torch.sigmoid` et seuiller à `threshold`
2. Calculer `intersection = (prediction * target).sum()`
3. Calculer `union = prediction.sum() + target.sum() - intersection`
4. Retourner `(intersection + 1e-6) / (union + 1e-6)`

In [ ]:
def iou_score(prediction, target, threshold=0.5):
    # 1. Appliquer sigmoid et seuiller
    # 2. Calculer l'intersection et l'union
    # 3. Retourner (intersection + 1e-6) / (union + 1e-6)
    return None  # À compléter

## Les fonctions d'entraînement

In [ ]:
def step(model, optimizer, criterion, x, y):
    """Effectue une étape d'entraînement sur un batch."""
    model.train()
    optimizer.zero_grad()

    prediction = model(x)
    loss = criterion(prediction, y)

    loss.backward()
    optimizer.step()

    iou = iou_score(prediction.detach().cpu(), y.cpu())
    return model, loss.item(), iou

In [ ]:
def fit(model, optimizer, criterion, epochs, trainloader, valloader):
    history = {'train_loss': [], 'val_loss': [], 'train_iou': [], 'val_iou': []}

    for epoch in range(epochs):
        # --- Entraînement ---
        model.train()
        train_loss, train_iou = 0.0, 0.0

        for x, y in trainloader:
            x, y = x.to(device), y.to(device)
            model, loss, iou = step(model, optimizer, criterion, x, y)
            train_loss += loss
            train_iou  += iou

        # --- Validation ---
        model.eval()
        val_loss, val_iou = 0.0, 0.0

        with torch.no_grad():
            for x, y in valloader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                val_loss += criterion(pred, y).item()
                val_iou  += iou_score(pred.cpu(), y.cpu())

        n_tr, n_val = len(trainloader), len(valloader)
        history['train_loss'].append(train_loss / n_tr)
        history['val_loss'].append(val_loss  / n_val)
        history['train_iou'].append(train_iou  / n_tr)
        history['val_iou'].append(val_iou   / n_val)

        print(f"Epoch {epoch+1:02d}/{epochs} | "
              f"Train Loss: {train_loss/n_tr:.4f} | Val Loss: {val_loss/n_val:.4f} | "
              f"Train IoU: {train_iou/n_tr:.4f} | Val IoU: {val_iou/n_val:.4f}")

    return model, history

## Initialisez et entraînez votre modèle

Initialisez :
- `model` : un `UNet(n_channels=3, n_classes=1)` déplacé sur `device`
- `criterion` : une `DiceLoss()`
- `optimizer` : `torch.optim.Adam` avec learning rate `1e-3`

In [ ]:
model     = None  # À compléter
criterion = None  # À compléter
optimizer = None  # À compléter

Utilisez la fonction `fit` pour entraîner votre modèle pendant **15 epochs**.

In [ ]:
epochs = 15
model, history = None  # Utilisez la fonction fit

## Visualisation des performances

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train')
plt.plot(history['val_loss'],   label='Validation')
plt.xlabel('Epochs')
plt.ylabel('Dice Loss')
plt.title('Courbe de perte')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_iou'], label='Train')
plt.plot(history['val_iou'],   label='Validation')
plt.xlabel('Epochs')
plt.ylabel('IoU')
plt.title('Courbe IoU')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Visualisation des prédictions sur des images de validation
model.eval()
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
col_titles = ['Image', 'Masque réel', 'Prédiction', 'Superposition']
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=12, fontweight='bold')

with torch.no_grad():
    for i in range(4):
        img, mask = dataset_val[i * 30]
        pred = model(img.unsqueeze(0).to(device))
        pred_binary = (torch.sigmoid(pred) > 0.5).float().squeeze().cpu()

        img_display = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

        axes[i, 0].imshow(img_display)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(mask.squeeze(), cmap='gray')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(pred_binary, cmap='gray')
        axes[i, 2].axis('off')

        overlay = img_display.copy()
        pred_np = pred_binary.numpy()
        overlay[pred_np == 1] = overlay[pred_np == 1] * 0.4 + np.array([0.0, 0.8, 0.0]) * 0.6
        axes[i, 3].imshow(overlay.clip(0, 1))
        axes[i, 3].axis('off')

plt.tight_layout()
plt.show()